# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guided workflow for loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata.to_json()

print("Title: {}\nDescription: {}".format(metadata['name'], metadata['description']))
print("Published: {}".format(metadata.get('datePublished', 'N/A')))
print("Cite As: {}".format(metadata.get('citeAs', 'N/A')))

## 2. Data Overview
Review available record sets, fields, and their `@id`s. 

**Note:** All entities are referenced via their `@id` fields for consistency.

In [ ]:
# Get all record set IDs from metadata
record_sets_metadata = metadata.get('recordSet', [])
if not record_sets_metadata:
    print("No record sets found in metadata. Please check the dataset schema.")
else:
    print(f"Found {len(record_sets_metadata)} record sets:")
    for rs in record_sets_metadata:
        rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs
        print(f"RecordSet @id: {rs_id}")

        # Load sample records for overview
        try:
            sample_records = list(dataset.records(record_set=rs_id))
            if sample_records:
                print(f"  Fields in RecordSet (by @id): {list(sample_records[0].keys())}")
            else:
                print("  No records available.")
        except Exception as e:
            print(f"  Error loading records for {rs_id}: {e}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

_Each DataFrame is keyed by its record set `@id`._

In [ ]:
# Build a list of RecordSet @id values
record_set_ids = []
for rs in record_sets_metadata:
    rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs
    record_set_ids.append(rs_id)

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {record_set_id}")
        print(f"Columns (@id): {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Error extracting DataFrame for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. Use field and column `@id`

- Filter records based on a numeric field
- Normalize the numeric field
- Group by a categorical field

_**Replace values below with actual `@id` values as found in your dataset overview above._

In [ ]:
# Example: Use the first available record set and numeric field

if len(record_set_ids) > 0:
    # Choose the first record set for demonstration
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Select a numeric field (replace with actual @id)
    # Attempt to infer numeric columns
    numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")

        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field
        cat_fields = [col for col in df.columns if df[col].dtype == 'object']
        if cat_fields:
            group_field = cat_fields[0]
            print(f"Grouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No categorical fields found for grouping.")
    else:
        print("No numeric fields found in RecordSet for EDA.")
else:
    print("No RecordSet available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

_Choose numeric and group fields as explored above._

In [ ]:
# Plot numeric field distribution for filtered records
if "filtered_df" in locals() and not filtered_df.empty and "numeric_field" in locals():
    plt.figure(figsize=(8, 4))
    plt.hist(filtered_df[numeric_field], bins=10, color='skyblue', edgecolor='k')
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if "group_field" in locals():
        plt.figure(figsize=(8, 4))
        filtered_df.boxplot(column=numeric_field, by=group_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset provides detailed clinical and molecular characteristics of cancer survivors with second primary colorectal cancer.
- `mlcroissant` enables easy loading and metadata-driven analysis referencing entities by their `@id`.
- Numeric and group fields can be extracted, filtered, normalized, and visualized for exploratory insights.

Further extensions may include more specific domain filtering, advanced normalization, or machine learning modeling using the processed DataFrames.